# UC-12 — FOCUS FinOps seed (Fabric)

Leest `Files/uc12_seed/focus-test.csv` (553 rijen FOCUS-spec billing data) en
bouwt de 5 marts in `Tables/`:

  - `mart_uc12_focus_spend_monthly`
  - `mart_uc12_focus_service_breakdown`
  - `mart_uc12_focus_commitment_utilization`
  - `mart_uc12_focus_top_resources`
  - `mart_uc12_focus_savings`

Spiegelt 1:1 de dbt staging + mart SQL uit `dbt/models/{staging,marts}/uc12_focus_finops/`.
Geen externe afhankelijkheden — alles via Spark SQL.

**Idempotent**: `mode('overwrite')` op elke target. Dezelfde input → identieke output.

SYNTHETIC DATA — UWV REFERENCE PLATFORM — NOT FOR REAL USE.

In [ ]:
# Parameters — injected door fabric_helpers.trigger_notebook.
workspace_id = "878307a8-e99a-4b9d-91c4-7b8fc457183b"
lakehouse_id = "10af248d-ba98-48c6-94db-fd2289b4f4a2"
csv_relative_path = "uc12_seed/focus-test.csv"

In [ ]:
# Pad-helpers — Files/ voor de CSV, Tables/ voor de marts.
BASE = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"
FILES_ROOT = f"{BASE}/Files"
TABLES_ROOT = f"{BASE}/Tables"
CSV_PATH = f"{FILES_ROOT}/{csv_relative_path}"
print("reading from:", CSV_PATH)

In [ ]:
# Bronze — laad CSV raw (PascalCase) + audit-kolommen.
from pyspark.sql import functions as F

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss'Z'")
    .csv(CSV_PATH)
)
print(f"bronze raw: {df_raw.count()} rijen × {len(df_raw.columns)} kolommen")
df_raw = (
    df_raw
    .withColumn("ingestion_ts", F.current_timestamp())
    .withColumn("source_file", F.lit(csv_relative_path))
    .withColumn("event_date", F.current_date())
)
df_raw.createOrReplaceTempView("bronze_focus")

In [ ]:
# Silver staging — mirror van dbt stg_focus_billing.
#   - PascalCase → snake_case
#   - lower(Provider)
#   - billing_month = date_trunc('month', BillingPeriodStart)
#   - tags JSON → tag_environment / tag_application / tag_cost_center
#   - savings_amount = coalesce(ListCost, EffectiveCost) - EffectiveCost
stg = spark.sql("""
SELECT
  BillingAccountId                                  AS billing_account_id,
  BillingAccountName                                AS billing_account_name,
  SubAccountId                                      AS sub_account_id,
  SubAccountName                                    AS sub_account_name,
  BillingPeriodStart                                AS billing_period_start,
  BillingPeriodEnd                                  AS billing_period_end,
  ChargePeriodStart                                 AS charge_period_start,
  ChargePeriodEnd                                   AS charge_period_end,
  CAST(date_trunc('MONTH', BillingPeriodStart) AS DATE) AS billing_month,
  ChargeCategory                                    AS charge_category,
  ChargeSubcategory                                 AS charge_subcategory,
  ChargeDescription                                 AS charge_description,
  ChargeFrequency                                   AS charge_frequency,
  BillingCurrency                                   AS billing_currency,
  CAST(BilledCost AS DOUBLE)                        AS billed_cost,
  CAST(EffectiveCost AS DOUBLE)                     AS effective_cost,
  CAST(ListCost AS DOUBLE)                          AS list_cost,
  CAST(ListUnitPrice AS DOUBLE)                     AS list_unit_price,
  CAST(COALESCE(ListCost, EffectiveCost) - EffectiveCost AS DOUBLE) AS savings_amount,
  PricingCategory                                   AS pricing_category,
  CAST(PricingQuantity AS DOUBLE)                   AS pricing_quantity,
  PricingUnit                                       AS pricing_unit,
  CAST(UsageQuantity AS DOUBLE)                     AS usage_quantity,
  UsageUnit                                         AS usage_unit,
  CommitmentDiscountCategory                        AS commitment_discount_category,
  CommitmentDiscountId                              AS commitment_discount_id,
  CommitmentDiscountName                            AS commitment_discount_name,
  CommitmentDiscountType                            AS commitment_discount_type,
  LOWER(Provider)                                   AS provider,
  Publisher                                         AS publisher,
  InvoiceIssuer                                     AS invoice_issuer,
  ServiceCategory                                   AS service_category,
  ServiceName                                       AS service_name,
  SkuId                                             AS sku_id,
  SkuPriceId                                        AS sku_price_id,
  ResourceId                                        AS resource_id,
  ResourceName                                      AS resource_name,
  ResourceType                                      AS resource_type,
  Region                                            AS region,
  AvailabilityZone                                  AS availability_zone,
  Tags                                              AS tags_raw,
  get_json_object(Tags, '$.environment')            AS tag_environment,
  get_json_object(Tags, '$.application')            AS tag_application,
  get_json_object(Tags, '$.cost_center')            AS tag_cost_center
FROM bronze_focus
""")
stg.createOrReplaceTempView("stg_focus_billing")
print(f"silver stg: {stg.count()} rijen × {len(stg.columns)} kolommen")

In [ ]:
# Mart 1 — mart_uc12_focus_spend_monthly.
mart1 = spark.sql("""
SELECT
  billing_month, provider, service_category, charge_category, billing_currency,
  SUM(billed_cost)                                AS billed_cost,
  SUM(effective_cost)                             AS effective_cost,
  SUM(COALESCE(list_cost, effective_cost))        AS list_cost,
  SUM(savings_amount)                             AS savings_amount,
  SUM(usage_quantity)                             AS usage_quantity,
  COUNT(*)                                        AS n_charges,
  COUNT(DISTINCT resource_id)                     AS n_resources,
  COUNT(DISTINCT sub_account_id)                  AS n_sub_accounts
FROM stg_focus_billing
GROUP BY billing_month, provider, service_category, charge_category, billing_currency
""")
mart1.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{TABLES_ROOT}/mart_uc12_focus_spend_monthly")
print(f"mart_uc12_focus_spend_monthly: {mart1.count()} rijen")

In [ ]:
# Mart 2 — mart_uc12_focus_service_breakdown (alleen Usage-charges).
mart2 = spark.sql("""
SELECT
  billing_month, provider, service_category, service_name, region,
  COALESCE(tag_environment, 'unknown')            AS environment,
  COALESCE(tag_application, 'unknown')            AS application,
  billing_currency,
  SUM(effective_cost)                             AS effective_cost,
  SUM(billed_cost)                                AS billed_cost,
  SUM(usage_quantity)                             AS usage_quantity,
  COUNT(DISTINCT resource_id)                     AS n_resources,
  COUNT(*)                                        AS n_charges
FROM stg_focus_billing
WHERE charge_category = 'Usage'
GROUP BY billing_month, provider, service_category, service_name, region,
         COALESCE(tag_environment, 'unknown'),
         COALESCE(tag_application, 'unknown'),
         billing_currency
""")
mart2.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{TABLES_ROOT}/mart_uc12_focus_service_breakdown")
print(f"mart_uc12_focus_service_breakdown: {mart2.count()} rijen")

In [ ]:
# Mart 3 — mart_uc12_focus_commitment_utilization.
mart3 = spark.sql("""
WITH agg AS (
  SELECT
    billing_month, provider, service_category, billing_currency,
    SUM(CASE WHEN pricing_category = 'Committed' THEN effective_cost ELSE 0 END) AS committed_spend,
    SUM(CASE WHEN pricing_category = 'Standard'  THEN effective_cost ELSE 0 END) AS on_demand_spend,
    SUM(CASE WHEN pricing_category = 'Dynamic'   THEN effective_cost ELSE 0 END) AS dynamic_spend,
    SUM(effective_cost)                                                          AS total_spend,
    SUM(savings_amount)                                                          AS savings_amount,
    COUNT(DISTINCT CASE WHEN pricing_category = 'Committed'
                        THEN commitment_discount_id END)                         AS n_active_commitments
  FROM stg_focus_billing
  WHERE charge_category = 'Usage'
  GROUP BY billing_month, provider, service_category, billing_currency
)
SELECT
  *,
  CASE WHEN total_spend > 0 THEN committed_spend / total_spend ELSE 0 END
    AS commitment_coverage_pct,
  CASE WHEN (committed_spend + on_demand_spend + dynamic_spend) > 0
       THEN savings_amount / (committed_spend + on_demand_spend + dynamic_spend + savings_amount)
       ELSE 0 END
    AS effective_discount_pct
FROM agg
""")
mart3.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{TABLES_ROOT}/mart_uc12_focus_commitment_utilization")
print(f"mart_uc12_focus_commitment_utilization: {mart3.count()} rijen")

In [ ]:
# Mart 4 — mart_uc12_focus_top_resources met rank_in_month.
mart4 = spark.sql("""
WITH agg AS (
  SELECT
    billing_month, provider,
    COALESCE(resource_id, CONCAT('account-level:', sub_account_id)) AS resource_id,
    MAX(resource_name)                              AS resource_name,
    MAX(resource_type)                              AS resource_type,
    MAX(service_name)                               AS service_name,
    MAX(service_category)                           AS service_category,
    MAX(region)                                     AS region,
    MAX(COALESCE(tag_application, 'unknown'))       AS application,
    MAX(COALESCE(tag_environment, 'unknown'))       AS environment,
    MAX(COALESCE(tag_cost_center, 'unknown'))       AS cost_center,
    MAX(sub_account_name)                           AS sub_account_name,
    billing_currency,
    SUM(effective_cost)                             AS effective_cost,
    SUM(billed_cost)                                AS billed_cost,
    SUM(usage_quantity)                             AS usage_quantity,
    MAX(usage_unit)                                 AS usage_unit,
    COUNT(*)                                        AS n_charges
  FROM stg_focus_billing
  WHERE charge_category = 'Usage'
  GROUP BY billing_month, provider,
           COALESCE(resource_id, CONCAT('account-level:', sub_account_id)),
           billing_currency
)
SELECT
  *,
  ROW_NUMBER() OVER (PARTITION BY billing_month, provider
                     ORDER BY effective_cost DESC) AS rank_in_month
FROM agg
""")
mart4.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{TABLES_ROOT}/mart_uc12_focus_top_resources")
print(f"mart_uc12_focus_top_resources: {mart4.count()} rijen")

In [ ]:
# Mart 5 — mart_uc12_focus_savings.
mart5 = spark.sql("""
SELECT
  billing_month, provider, pricing_category, service_category, charge_category,
  billing_currency,
  SUM(COALESCE(list_cost, effective_cost))        AS list_cost,
  SUM(effective_cost)                             AS effective_cost,
  SUM(savings_amount)                             AS savings_amount,
  CASE WHEN SUM(COALESCE(list_cost, effective_cost)) > 0
       THEN SUM(savings_amount) / SUM(COALESCE(list_cost, effective_cost))
       ELSE 0 END                                 AS savings_pct,
  COUNT(*)                                        AS n_charges,
  COUNT(DISTINCT resource_id)                     AS n_resources
FROM stg_focus_billing
GROUP BY billing_month, provider, pricing_category, service_category, charge_category,
         billing_currency
""")
mart5.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{TABLES_ROOT}/mart_uc12_focus_savings")
print(f"mart_uc12_focus_savings: {mart5.count()} rijen")

In [ ]:
print("UC-12 seed klaar — 5 marts geschreven naar Tables/ in uc11_lakehouse.")